# YOISAKI ENGINE v1.7.0 — IMG2IMG EDITION

Для YOISAKI AI STUDIO v1.7.0. Выберите T4 GPU, выполните подготовку и запустите движок через существующую панель.

**v1.7.0 — Img2Img:** настоящий Image-to-Image (вкладка IMG2IMG на сайте): исходник, Denoise, LoRA, seed, sampler/scheduler, batch, Regional Control и ControlNet. Конвейер Img2Img строится через `from_pipe()` поверх уже загруженных компонентов SDXL — вторая копия модели не загружается; после задания возвращается базовый конвейер. **Img2Img на настоящем T4 не проверен в среде разработки.**

OpenPose с выбором препроцессора позы — OpenPose Classic (человеческое тело) или SDPose-OOD (стилизованные, аниме, фурри, сложные позы), Canny (OpenCV), Depth (Depth Anything V2 Small), готовые карты, preview через сайт. Модели скачиваются лениво; первый запуск требует интернета. Предобработка OpenPose Classic / Canny / Depth выполняется на CPU; SDPose-OOD — на GPU в fp16 и полностью выгружается до генерации. Regional Control + ControlNet одновременно отклоняются.

Сохранены SDXL/Illustrious, LoRA, длинные промпты, очередь, CPU offload и диагностика памяти. ControlNet выгружается после каждой задачи, базовые компоненты переиспользуются.

**GPU inference, визуальная совместимость конкретных Illustrious checkpoint и RAM/VRAM T4 НЕ ПРОВЕРЕНЫ в среде разработки.** Автоматические OpenPose Classic / SDPose-OOD / Depth требуют проверки скачивания и исполнения моделей в Colab; Canny проверен на CPU.

Предыдущие заметки о RAM ниже сохранены как история исправлений. Актуальная версия движка — v1.7.0.


**v1.6.1 — только исправления ошибок v1.6.0:** Strength 0/1/2, сохранение fp16 при переходах ControlNet, fp16-вариант Canny, освобождение ControlNet без копирования в RAM, возобновляемая предобработка, очистка таблицы заданий, бинарные карты Canny, отбраковка однородной глубины. Подробно: `YOISAKI_V1.6.1_STABILITY_REPORT.md`. Реальные RAM/VRAM на T4 по-прежнему требуют проверки.

**v1.6.2 — SDPose-OOD:** новый препроцессор позы для существующего OpenPose ControlNet (`teemosliang/SDPose-Body`, загружается только на время подготовки карты; первая подготовка скачивает веса и идёт дольше). При недоступном Hugging Face можно указать локальную копию: `os.environ['YOISAKI_SDPOSE_REPO']='/content/drive/MyDrive/SDPose-Body'` до запуска движка. Подробно: `YOISAKI_V1.6.2_SDPOSE_REPORT.md`. Загрузка настоящих весов SDPose-OOD и его VRAM на T4 в среде разработки не проверены.

## История исправлений памяти v1.5.2 (сохранено в v1.7.0)

Точечная коррекция поведения glibc allocator при CPU Offload. Реальные две генерации на T4 **ещё не проверены**; это не обещание устранённой утечки. Pipeline, качество и offload сохранены.

**Откройте в новой среде T4.** Настройка применяется при старте отдельного worker; уже запущенный сервер её не получит. Выполните подготовку, запустите движок и исходную панель диагностики (ячейка 3). Выберите тот же Illustrious XL, 2 изображения, 768×768, 20 шагов, CFG 5, batch 1, seed 123, без LoRA/Regional/ControlNet. На первом прогоне оставьте подробное сканирование и malloc_trim выключенными.

Диагностика не запускает следующее задание при свободной RAM менее 4 GiB (или более высокого выбранного порога). Это не защита от всех скачков внутри текущей операции и **не доказательство исправления**. Не понижайте порог ради двух изображений. Скачайте ZIP и при safety_stop.

Сравнивайте USS/PSS после очистки и MemAvailable; новые точки `before_unet_cpu_offload` / `after_unet_cpu_offload` отделяют возврат весов от декодирования VAE. В environment.json поле `malloc_mmap_threshold_requested` обычно должно быть `131072`; это запрос настройки, не доказательство работы allocator. В среде с LD_PRELOAD или glibc malloc tunables автоматическая настройка пропускается.

Результаты сохраняются прежним способом. Генератор не перезагружает checkpoint между заданиями. Для научного A/B можно перед запуском worker установить `os.environ['YOISAKI_RAM_FIX']='0'` в новой среде; это отключает только новую настройку allocator и не снимает защиту. Повторять опасный исходный прогон не требуется.


**Диагностическое исправление v1.5.2.1:** синхронизирована проверка версии `/health` в `diagnostics/runner.py` с реальным сервером 1.5.2. Это не дополнительное исправление RAM. Обычный код генерации не менялся.


In [ ]:
# @title 🚀 1. ПОДГОТОВИТЬ ПАНЕЛЬ { display-mode: "form" }
Подключить_Google_Drive = True # @param {type:"boolean"}
import base64, hashlib, io, zipfile, sys, subprocess, traceback, html
from pathlib import Path
from IPython.display import display, HTML
_ROOT = Path('/content/yoisaki_engine')
_PACKAGE = '...'
_EXPECTED = 'f50bf634cdd6c0fa00cab11cc35a26e45733e0ade83903581f05b0f2deea8d7c'
def _prepare():
    global yoisaki_panel
    display(HTML('<div style="padding:16px;background:#202228;color:#ffc66b;border-radius:12px"><b>🦊 YOISAKI ENGINE v1.7.0</b><br>⏳ Подготовка окружения. Обычно несколько минут; не запускайте ячейку повторно.</div>'))
    if 'yoisaki_panel' in globals():yoisaki_panel.close()
    raw=base64.b64decode(_PACKAGE)
    if hashlib.sha256(raw).hexdigest()!=_EXPECTED:raise ValueError('Файлы движка повреждены. Заново откройте исходный блокнот v1.7.0.')
    _ROOT.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        for info in z.infolist():
            target=(_ROOT/info.filename).resolve()
            if not target.is_relative_to(_ROOT.resolve()) or info.is_dir() or (info.external_attr>>16)&0o170000==0o120000:raise ValueError('Небезопасный файл в пакете')
        z.extractall(_ROOT)
    if not (_ROOT/'worker/direct_server.py').is_file():raise FileNotFoundError('Файлы движка не найдены. Откройте исходный блокнот повторно.')
    # Ensure rerunning uses exactly this version's modules.
    for name in list(sys.modules):
        if name in ('panel','bootstrap','worker','inference','diagnostics') or name.startswith(('worker.','inference.','diagnostics.')):sys.modules.pop(name,None)
    sys.path.insert(0,str(_ROOT))
    from bootstrap import install
    install()
    try:import ipywidgets
    except ImportError:
        with (_ROOT/'widgets-install.log').open('w') as log:
            subprocess.run([sys.executable,'-m','pip','install','ipywidgets>=8.1,<9'],check=True,stdout=log,stderr=subprocess.STDOUT)
    from google.colab import output
    output.enable_custom_widget_manager()
    from panel import Panel
    yoisaki_panel=Panel()
    yoisaki_panel.show()
    if Подключить_Google_Drive:yoisaki_panel.action(yoisaki_panel.mount)
try:
    _prepare()
except Exception:
    detail=traceback.format_exc()
    log=_ROOT/'setup.log'
    if log.exists():detail+='\n'+log.read_text(errors='replace')[-18000:]
    display(HTML('<div style="padding:12px;border:1px solid #d69332">⚠ Подготовка не завершена. Проверьте интернет и выберите T4 GPU. Повторите эту ячейку. Если ошибка повторяется — отключите и удалите среду выполнения, затем запустите блокнот заново.</div><details><summary>🔧 Техническая диагностика</summary><pre style="white-space:pre-wrap">'+html.escape(detail)+'</pre></details>'))


In [ ]:
# @title 🛠 Резервное управление — только если кнопки не отображаются { display-mode: "form" }
Действие = "Обновить статус" # @param ["Обновить статус", "Запустить", "Остановить", "Подключить Drive", "Диагностика"]
if 'yoisaki_panel' not in globals():
    print('Сначала запустите «ПОДГОТОВИТЬ ПАНЕЛЬ».')
else:
    actions={'Обновить статус':yoisaki_panel.refresh,'Запустить':yoisaki_panel.start,'Остановить':yoisaki_panel.stop,'Подключить Drive':yoisaki_panel.mount,'Диагностика':yoisaki_panel.diagnose}
    yoisaki_panel.action(actions[Действие])
    display(HTML(yoisaki_panel.status.value))
    if yoisaki_panel.error.value:display(HTML(yoisaki_panel.error.value))
    if Действие=='Диагностика':display(HTML(yoisaki_panel.diagnostics.value))
    if yoisaki_panel.controller.verified:
        display(HTML('<p>Удерживайте ссылку, выделите целиком и скопируйте:</p><pre style="white-space:pre-wrap;overflow-wrap:anywhere">'+html.escape(yoisaki_panel.controller.url)+'</pre>'))


## Диагностика памяти 1.5.2

Автоматические замеры: `MyDrive/YOISAKI_AI_STUDIO/diagnostics/memory.jsonl` (байты; ротация около 16 MiB суммарно). Токены и prompts не записываются. Низкая VRAM после генерации ожидаема при CPU offload и сама по себе не доказывает утечку.

Дополнительный реальный тест: после запуска панели добавьте кодовую ячейку с примером ниже. Не отправляйте задания с сайта одновременно. Подставьте ID checkpoint из каталога; ID LoRA необязательны. Сначала выполняются 10 настоящих Text2Img без LoRA; затем только сценарии с указанными LoRA. Результаты и summary.jsonl сохраняются в diagnostics/stress на Drive. Это не выполненный заранее тест и не имитация.

```python
from panel import fetch
from stress_memory import run
catalog = fetch("http://127.0.0.1:8765/models", yoisaki_panel.controller.token)["models"]
[(m["id"], m["kind"], m["name"]) for m in catalog]
```

После выбора ID:

```python
run(yoisaki_panel.controller.token, checkpoint_id="ID_CHECKPOINT",
    character_id="", second_character_id="", style_id="",
    trigger_a="", trigger_b="", width=768, height=768, steps=20)
```

Не печатайте token. Для замеров на другом разрешении повторите весь тест с фиксированными width/height. Полный отчёт: MEMORY_DIAGNOSTICS_REPORT_RU.md в архиве сайта.


## 🔬 YOISAKI MEMORY PROFILER v1.0

**Профессиональная диагностика RAM / VRAM (без изменения генератора).** После «ПОДГОТОВИТЬ ПАНЕЛЬ» и «ЗАПУСТИТЬ ДВИЖОК» запустите ячейку ниже. Выберите checkpoint и нажмите **🔬 ЗАПУСТИТЬ ДИАГНОСТИКУ**.

По умолчанию: **2 реальные последовательные генерации**, 768×768, 20 шагов, batch 1, **LoRA нет**, **Regional Control OFF**, 1 проба в секунду. Диагностическая ячейка общается с существующим сервером через локальный авторизованный API, без изменения UI сайта. Не отправляйте параллельно задания с сайта.

**Безопасность:** тест остановится при опасно низкой системной RAM или слишком большом RSS рабочего процесса; такой останов нельзя считать успешной проверкой стабильности. Точки и график пишутся в Google Drive в реальном времени. При сбое остаётся каталог `.partial`, который можно восстановить.

Архив создаётся на Drive по пути `MyDrive/YOISAKI_AI_STUDIO/diagnostics/reports/<уникальное_имя>.zip`; кнопка «📥 СКАЧАТЬ ОТЧЁТ» скачивает локальную копию с именем `YOISAKI_MEMORY_REPORT.zip`.

Указанные замеры отражают состояние реально исполнявшегося процесса. Версии PyTorch/Diffusers, состав model hooks, список компонентов и недоступные метрики включаются в ZIP. Результаты не включают промпты, изображения, секреты и содержимое тензоров.

*Старый пример `stress_memory.py` из базового блокнота оставлен для совместимости, но для поиска причины роста RAM используйте панель ниже. Подробные CPU-тензоры и malloc_trim включайте лишь при необходимости.*

In [ ]:
# @title 🔬 3. ЗАПУСТИТЬ ПАНЕЛЬ ДИАГНОСТИКИ { display-mode: "form" }
# Отдельный компактный виджет. Основная GPU-панель и сервер остаются без изменений.
try:
    if 'yoisaki_panel' not in globals():
        raise RuntimeError('Сначала выполните «ПОДГОТОВИТЬ ПАНЕЛЬ» выше.')
    from diagnostics.ui import NotebookDiagnostics
    if 'yoisaki_memory_ui' in globals():
        try:yoisaki_memory_ui.widget.close()
        except Exception:pass
    yoisaki_panel.refresh()
    yoisaki_memory_ui=NotebookDiagnostics(yoisaki_panel)
    yoisaki_memory_ui.show()
    print('📁 Сохранение логов: MyDrive/YOISAKI_AI_STUDIO/diagnostics/reports/')
    print('Если Colab оборвался, выполните после новой подготовки:')
    print('from diagnostics.runner import recover_partial_reports; recover_partial_reports()')
except Exception as _profiler_exc:
    print('⚠ Диагностика не готова:',type(_profiler_exc).__name__,str(_profiler_exc))


## Аудит промпта
После первой генерации запустите следующую ячейку. Поддерживается до 16 окон CLIP; превышение отклоняется, текст не обрезается. Синтаксис `(tag:1.2)` и `BREAK` не интерпретируется как WebUI-команды. Clip Skip фиксирован: предпоследний слой SDXL. Pooled embedding берётся из первого окна второго энкодера. Совпадение с внутренним кодированием Civitai не доказано.


In [ ]:
# @title 🔎 4. Проверить токены и параметры запроса (необязательно)
# Textarea contents are not written to files by this tool.
import ipywidgets as widgets, requests, json
from IPython.display import display, clear_output
_audit_pos=widgets.Textarea(description='Positive',layout=widgets.Layout(width='100%',height='120px'))
_audit_neg=widgets.Textarea(description='Negative',layout=widgets.Layout(width='100%',height='90px'))
_audit_text=widgets.Checkbox(value=False,description='Показать отброшенные старым движком слова (приватный текст в выводе)')
_audit_id=widgets.Text(description='Job ID',layout=widgets.Layout(width='100%'))
_audit_button=widgets.Button(description='Проверить токены',button_style='info')
_trace_button=widgets.Button(description='Параметры задания')
_audit_output=widgets.Output()
def _audit_call(_button):
    with _audit_output:
        clear_output(wait=True)
        try:
            token=yoisaki_panel.controller.token
            if not token:raise RuntimeError('Сначала запустите движок и выполните одну генерацию.')
            headers={'Authorization':'Bearer '+token}
            if _button is _audit_button:
                response=requests.post('http://127.0.0.1:8765/prompt-audit',headers=headers,json={'prompt':_audit_pos.value,'negative_prompt':_audit_neg.value,'include_text':_audit_text.value},timeout=30)
                if response.status_code!=200:raise RuntimeError(response.json().get('detail','Ошибка аудита'))
                value=response.json()
            else:
                import re
                ident=_audit_id.value.strip()
                if not re.fullmatch('[a-f0-9-]{36}',ident):raise RuntimeError('Вставьте полный ID задания из очереди сайта.')
                response=requests.get('http://127.0.0.1:8765/jobs/'+ident,headers=headers,timeout=30)
                if response.status_code!=200:raise RuntimeError('Задание не найдено в текущей сессии.')
                value=response.json().get('diagnostics') or {'status':'Диагностика появится после успешной генерации.'}
            print(json.dumps(value,ensure_ascii=False,indent=2))
        except Exception as exc:
            # Never print requests exceptions: they can contain credentials/URLs.
            print(str(exc) if isinstance(exc,RuntimeError) else 'Не удалось связаться с локальным движком.')
_audit_button.on_click(_audit_call);_trace_button.on_click(_audit_call)
display(widgets.VBox([widgets.HTML('Аудит использует токенизаторы последней загруженной модели. По умолчанию выводит только числа. Галочка раскрывает текст в выводе Colab; очистите вывод перед публикацией блокнота.'),_audit_pos,_audit_neg,_audit_text,_audit_button,_audit_id,_trace_button,_audit_output]))
